# NASA C-MAPSS EDA 

EDA for sensor distributions, lifecycle plots, correlation matrix, useful vs useless sensors and lifetime distributions.

In [ ]:
# Imports
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. File discovery and path setup

In [ ]:

BASE_DIR = Path.cwd()
print("Running from:", BASE_DIR)

def resolve_file(stem):
    candidates = [
        BASE_DIR / f"{stem}.txt",
        BASE_DIR / stem,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find file for stem: {stem}")

expected = [
    "train_FD001", "train_FD002", "train_FD003", "train_FD004",
    "test_FD001", "test_FD002", "test_FD003", "test_FD004",
    "RUL_FD001", "RUL_FD002", "RUL_FD003", "RUL_FD004"
]

resolved_files = {}
for stem in expected:
    try:
        resolved_files[stem] = resolve_file(stem)
    except FileNotFoundError:
        resolved_files[stem] = None

pd.DataFrame({
    "file_key": list(resolved_files.keys()),
    "path": [str(v) if v is not None else "MISSING" for v in resolved_files.values()],
    "exists": [v is not None for v in resolved_files.values()]
})

## 2. Column schema and loaders

In [ ]:

index_cols = ["engine_id", "cycle"]
op_cols = [f"op{i}" for i in range(1, 4)]
sensor_cols = [f"s{i}" for i in range(1, 22)]
all_cols = index_cols + op_cols + sensor_cols

def load_cmapss_table(path, columns=all_cols):
    df = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        engine="python"
    )
    # Drop fully empty trailing columns if present
    df = df.dropna(axis=1, how="all")
    df.columns = columns[:df.shape[1]]
    return df

def load_rul_file(path):
    rul = pd.read_csv(path, sep=r"\s+", header=None, engine="python")
    rul = rul.dropna(axis=1, how="all")
    rul.columns = ["RUL"]
    rul["engine_id"] = np.arange(1, len(rul) + 1)
    return rul[["engine_id", "RUL"]]

def add_dataset_id(df, dataset_id):
    out = df.copy()
    out["dataset_id"] = dataset_id
    return out

## 3. Load all train/test datasets

In [ ]:
train_sets = {}
test_sets = {}
rul_sets = {}

for fd in ["FD001", "FD002", "FD003", "FD004"]:
    train_sets[fd] = add_dataset_id(load_cmapss_table(resolved_files[f"train_{fd}"]), fd)
    test_sets[fd] = add_dataset_id(load_cmapss_table(resolved_files[f"test_{fd}"]), fd)
    rul_sets[fd] = load_rul_file(resolved_files[f"RUL_{fd}"])

summary_rows = []
for fd in ["FD001", "FD002", "FD003", "FD004"]:
    summary_rows.append({
        "dataset_id": fd,
        "train_rows": len(train_sets[fd]),
        "test_rows": len(test_sets[fd]),
        "train_engines": train_sets[fd]["engine_id"].nunique(),
        "test_engines": test_sets[fd]["engine_id"].nunique(),
        "rul_count": len(rul_sets[fd]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 4. Combine datasets and create training RUL target

In [ ]:
train_df = pd.concat([train_sets[k] for k in ["FD001", "FD002", "FD003", "FD004"]], ignore_index=True)
test_df = pd.concat([test_sets[k] for k in ["FD001", "FD002", "FD003", "FD004"]], ignore_index=True)

max_cycle = train_df.groupby(["dataset_id", "engine_id"])["cycle"].max().rename("max_cycle")
train_df = train_df.merge(max_cycle, on=["dataset_id", "engine_id"], how="left")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycle"]
train_df.drop(columns=["max_cycle"], inplace=True)

train_df.shape, test_df.shape

In [ ]:
train_df.head()

## 5. Basic overview

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train engines:", train_df.groupby("dataset_id")["engine_id"].nunique().to_dict())
print("Test engines :", test_df.groupby("dataset_id")["engine_id"].nunique().to_dict())

## 6. Lifetime distributions

For the training sets, each engine runs until failure.  
Engine lifetime here means the maximum observed cycle per engine.

In [ ]:
train_lifetimes = (
    train_df.groupby(["dataset_id", "engine_id"])["cycle"]
    .max()
    .reset_index(name="lifetime")
)

test_prefix_lengths = (
    test_df.groupby(["dataset_id", "engine_id"])["cycle"]
    .max()
    .reset_index(name="observed_test_length")
)

display(train_lifetimes.groupby("dataset_id")["lifetime"].describe())

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=train_lifetimes, x="lifetime", hue="dataset_id", bins=30, kde=True, element="step")
plt.title("Training engine lifetime distribution by dataset")
plt.xlabel("Lifetime (cycles)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_lifetimes, x="dataset_id", y="lifetime")
plt.title("Training lifetime distribution by dataset")
plt.xlabel("Dataset")
plt.ylabel("Lifetime (cycles)")
plt.show()

## 7. Sensor variance analysis: useful vs useless sensors

For identifying:

- nearly constant sensors
- very low-variance sensors
- sensors that visibly change with degradation

Low-variance or constant sensors are not useful for modeling.

In [ ]:
sensor_variance = train_df[sensor_cols].var().sort_values(ascending=False).rename("variance").reset_index()
sensor_variance.columns = ["sensor", "variance"]
sensor_variance

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=sensor_variance, x="sensor", y="variance")
plt.xticks(rotation=90)
plt.title("Sensor variance across combined training data")
plt.xlabel("Sensor")
plt.ylabel("Variance")
plt.show()

In [ ]:
low_variance_threshold = sensor_variance["variance"].quantile(0.25)

low_variance_sensors = sensor_variance.query("variance <= @low_variance_threshold")["sensor"].tolist()
high_variance_sensors = sensor_variance.query("variance > @low_variance_threshold")["sensor"].tolist()

print("Low-variance sensors (candidate less-useful sensors):")
print(low_variance_sensors)
print("\nHigher-variance sensors (candidate useful sensors):")
print(high_variance_sensors)

## 8. Sensor distributions

Distributions for all 21 sensors, for identifying skewed sensors, narrow sensors and multimodal sensors.

In [ ]:
fig, axes = plt.subplots(7, 3, figsize=(18, 24))
axes = axes.flatten()

for ax, s in zip(axes, sensor_cols):
    sns.histplot(train_df[s], bins=40, kde=True, ax=ax)
    ax.set_title(s)

for ax in axes[len(sensor_cols):]:
    ax.axis("off")

plt.suptitle("Sensor distributions across combined training data", y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

## 9. Lifecycle plots

How does sensor values evolve over the lifecycle of engines?

Normalized lifecycle variable:

- `life_pct = cycle / engine_max_cycle`

This makes different engine lengths comparable.

In [ ]:
engine_max = train_df.groupby(["dataset_id", "engine_id"])["cycle"].max().rename("engine_max_cycle")
train_life = train_df.merge(engine_max, on=["dataset_id", "engine_id"], how="left")
train_life["life_pct"] = train_life["cycle"] / train_life["engine_max_cycle"]

train_life[["dataset_id", "engine_id", "cycle", "engine_max_cycle", "life_pct", "RUL"]].head()

In [ ]:
# Average sensor trajectory over normalized lifetime
life_bins = np.linspace(0, 1, 21)
train_life["life_bin"] = pd.cut(train_life["life_pct"], bins=life_bins, include_lowest=True)

avg_lifecycle = (
    train_life.groupby(["dataset_id", "life_bin"])[sensor_cols]
    .mean(numeric_only=True)
    .reset_index()
)

avg_lifecycle["life_bin_mid"] = avg_lifecycle["life_bin"].apply(lambda x: x.mid)
avg_lifecycle.head()

In [ ]:
top_dynamic_sensors = sensor_variance.head(9)["sensor"].tolist()

fig, axes = plt.subplots(3, 3, figsize=(18, 14), sharex=True)
axes = axes.flatten()

for ax, s in zip(axes, top_dynamic_sensors):
    for fd in ["FD001", "FD002", "FD003", "FD004"]:
        tmp = avg_lifecycle[avg_lifecycle["dataset_id"] == fd]
        ax.plot(tmp["life_bin_mid"], tmp[s], label=fd)
    ax.set_title(f"{s} vs normalized lifecycle")
    ax.set_xlabel("Normalized lifecycle")
    ax.set_ylabel("Average value")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 10. Correlation matrix

We compute the sensor-to-sensor correlation matrix on the training data.
This helps identify redundant sensors and strongly coupled sensor groups.

In [ ]:
corr = train_df[sensor_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Sensor correlation matrix")
plt.show()

In [ ]:
# Highest absolute off-diagonal correlations
corr_pairs = (
    corr.where(~np.eye(corr.shape[0], dtype=bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["sensor_1", "sensor_2", "correlation"]
corr_pairs["abs_corr"] = corr_pairs["correlation"].abs()
corr_pairs = corr_pairs.sort_values("abs_corr", ascending=False)

# Remove symmetric duplicates
corr_pairs["pair_key"] = corr_pairs.apply(lambda r: tuple(sorted([r["sensor_1"], r["sensor_2"]])), axis=1)
corr_pairs = corr_pairs.drop_duplicates("pair_key").drop(columns="pair_key")

corr_pairs.head(20)

## 11. Linear Relationship with RUL

This section gives a quick ranking of sensors by their linear association with the training RUL target.

In [ ]:
rul_corr = (
    train_df[sensor_cols + ["RUL"]]
    .corr()["RUL"]
    .drop("RUL")
    .sort_values(key=np.abs, ascending=False)
    .rename("corr_with_RUL")
    .reset_index()
)
rul_corr.columns = ["sensor", "corr_with_RUL"]
rul_corr

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=rul_corr, x="sensor", y="corr_with_RUL")
plt.xticks(rotation=90)
plt.title("Sensor correlation with training RUL")
plt.xlabel("Sensor")
plt.ylabel("Correlation with RUL")
plt.show()

## 12. Dataset-specific sensor means

This helps show how datasets differ overall, especially because some subsets contain more operating regimes and/or fault modes.

In [ ]:
dataset_sensor_means = train_df.groupby("dataset_id")[sensor_cols].mean()
plt.figure(figsize=(16, 6))
sns.heatmap(dataset_sensor_means, cmap="viridis")
plt.title("Mean sensor values by dataset")
plt.xlabel("Sensor")
plt.ylabel("Dataset")
plt.show()

## 13. Saving cleaned EDA summary tables

In [ ]:
eda_outputs = {
    "summary_df.csv": summary_df,
    "train_lifetimes.csv": train_lifetimes,
    "sensor_variance.csv": sensor_variance,
    "sensor_rul_correlation.csv": rul_corr
}

save_outputs = True 

if save_outputs:
    out_dir = BASE_DIR / "eda_outputs"
    out_dir.mkdir(exist_ok=True)
    for name, df in eda_outputs.items():
        df.to_csv(out_dir / name, index=False)
    print("Saved outputs to:", out_dir)
else:
    print("save_outputs=False, so no files were written.")